In [70]:
# combine CMIP6 FWI data from multiple models into a single netCDF file
import xarray as xr
import datetime
from pathlib import Path

In [71]:
dir = Path("/beegfs/CMIP6/arctic-cmip6/cmip6_fwi/cmip6_fwi/data_release")
# list nc files
files = list(dir.glob("*.nc"))
files.sort()

In [72]:
# if files are era5, split into a different list and drop from the main list
era5_files = [f for f in files if "era5" in f.name]
files = [f for f in files if "era5" not in f.name]

In [73]:
# open multiple files with xarray
# need to preprocess to include model name and year as coordinates
# example filename structure is cffdrs_<model>_<YYYY>.nc (e.g. cffdrs_CNRM-CM6-1-HR_2007.nc)

def pull_dims_from_source(ds):

    var = list(ds.data_vars)[0]  # just use first var
    src = ds[var].encoding["source"]
    fp_model_id = src.split("/")[-1].split("_")[1]
    # add model to dataset as dimensions using an array with one value
    ds = ds.expand_dims({"model": [fp_model_id]})
    return ds

def preprocess(ds):
    ds = pull_dims_from_source(ds)
    # drop global encoding and attributes that are not needed
    ds.encoding = {}
    ds.attrs = {}
    return ds



In [74]:
cmip6_ds = xr.open_mfdataset(files, combine="by_coords", parallel=True, preprocess=preprocess)
era5_ds = xr.open_mfdataset(era5_files, combine="by_coords", parallel=True, preprocess=preprocess)

In [75]:
# fix the time dimension in era5_ds to be consistent with cmip6_ds
# cmip6_ds time is at noon, era5_ds time is at 0 hour
era5_ds["time"] = [t + datetime.timedelta(hours=12) for t in era5_ds["time"].values]

In [76]:
# Combine cmip6_ds and era5_ds along the "model" dimension, aligning on time
combined_ds = xr.concat([cmip6_ds, era5_ds], dim="model", join="outer")
combined_ds = combined_ds.sortby("time")
combined_ds

/home/jdpaul3/miniconda3/envs/snap-geo/lib/python3.11/site-packages/xarray/core/indexing.py:1624: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/home/jdpaul3/miniconda3/envs/snap-geo/lib/python3.11/site-packages/xarray/core/indexing.py:1624: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.a

<xarray.Dataset> Size: 537GB
Dimensions:  (model: 5, time: 44165, lat: 178, lon: 569)
Coordinates:
  * time     (time) object 353kB 1979-01-01 12:00:00 ... 2099-12-31 12:00:00
  * lat      (lat) float64 1kB 35.0 35.25 35.5 35.75 ... 78.5 78.75 79.0 79.25
  * lon      (lon) float64 5kB -177.0 -176.8 -176.5 ... -35.5 -35.25 -35.0
  * model    (model) object 40B 'CNRM-CM6-1-HR' 'EC-Earth3-Veg' ... 'era5'
Data variables:
    ffmc     (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    dmc      (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    dc       (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    isi      (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    bui      (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>
    fwi      (model, time, lat, lon) float32 89GB dask.array<chunksize=(1, 365, 178, 569), meta=np.ndarray>

In [77]:
# test the dataset; era5 should have NAN for years outside 1979-2023
print(combined_ds["fwi"].sel(model="era5", lat=65, lon=-145, time="1980-01-01").values)
print(combined_ds["fwi"].sel(model="era5", lat=65, lon=-145, time="2080-01-01").values)


[0.9572839]


[nan]


In [ ]:
# pick a random file from the era5_files list and the cmip6_files list
# extract the year and model from the filename
# open the file and print original values for lat 65 lon -147 date 01-01
# then compare to the combined dataset values for the same lat lon date and model
import random

test_era5_file = random.choice(era5_files)
test_era5_year = int(test_era5_file.stem.split("_")[2])

test_cmip6_file = random.choice(files)  
test_cmip6_year = int(test_cmip6_file.stem.split("_")[2])
test_cmip6_model = test_cmip6_file.stem.split("_")[1]

era5_ds_single = xr.open_dataset(test_era5_file)
cmip6_ds_single = xr.open_dataset(test_cmip6_file)

print(f"Testing ERA5 file: {test_era5_file.name}")
print(f"Original ERA5 value: {era5_ds_single['fwi'].sel(lat=65, lon=-147, time=f'{test_era5_year}-01-01').values}")
print(f"Combined ERA5 value: {combined_ds['fwi'].sel(model='era5', lat=65, lon=-147, time=f'{test_era5_year}-01-01').values}")

print("\n\n")

print(f"Testing CMIP6 file: {test_cmip6_file.name}")
print(f"Original CMIP6 value: {cmip6_ds_single['fwi'].sel(lat=65, lon=-147, time=f'{test_cmip6_year}-01-01').values}")
print(f"Combined CMIP6 value: {combined_ds['fwi'].sel(model=test_cmip6_model, lat=65, lon=-147, time=f'{test_cmip6_year}-01-01').values}")




Testing ERA5 file: cffdrs_era5_2015.nc
Test year: 2015
Original ERA5 value: [0.89742106]


Combined ERA5 value: [0.89742106]



Testing CMIP6 file: cffdrs_CNRM-CM6-1-HR_2039.nc
Test year: 2039
Original CMIP6 value: [2.868363]
Combined CMIP6 value: [2.868363]


In [ ]:
# save combined_ds to netcdf
combined_ds.to_netcdf(dir / "cmip6_fwi_combined.nc", mode="w")